# SpaceX launch sites locations analysis with Folium 

The launch success rate may depend on many factors such as payload mass, orbit type, and so on. It may also depend on the location and proximities of a launch site, i.e., the initial position of rocket trajectories. Finding an optimal location for building a launch site certainly involves many factors and hopefully we could discover some of the factors by analyzing the existing launch site locations.

This lab contains the following tasks:

- TASK 1: Mark all launch sites on a map,
- TASK 2: Mark the success/failed launches for each site on the map,
- TASK 3: Calculate the distances between a chosen launch site to its proximities.

After completed the above tasks, you should be able to find some geographical patterns about launch sites.

Import libraries and prepare dataset

In [2]:
import pandas as pd
import folium

# Import folium MarkerCluster plugin
from folium.plugins import MarkerCluster

# Import folium MousePosition plugin
from folium.plugins import MousePosition

# Import folium DivIcon plugin
from folium.features import DivIcon

In [3]:
URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv'
spacex_df=pd.read_csv(URL)

Data wrangling

In [ ]:
# rename the columns so we wouldn't need to deal with the spaces and others: 

column_mapper = {'Launch Site':'LaunchSite','class':'Class','Landing Outcome':'LandingOutcome'}
spacex_df = spacex_df.rename(mapper=column_mapper, axis=1)
spacex_df.head()

In [ ]:
spacex_df["LaunchSite"].value_counts()

Basic map

In [218]:
# create a folium Map object, with an initial center location to be NASA Johnson Space Center at Houston, Texas:

nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

Circle & Marker

In [ ]:
# Create a blue circle at NASA Johnson Space Center's coordinate with a popup label showing its name

circle = folium.Circle(nasa_coordinate, radius=500, color='blue', fill=True).add_child(folium.Popup('NASA Johnson Space Center'))
site_map.add_child(circle)

# Create a blue circle at NASA Johnson Space Center's coordinate with a icon showing its name

marker = folium.map.Marker(nasa_coordinate, icon=DivIcon(icon_size=(20,20),icon_anchor=(0,0), html='<div style="font-size: 12; color:blue;"><b>%s</b></div>' % 'NASA JSC',))
site_map.add_child(marker)

Feature group

In [ ]:
# instantiate a feature group for the sites in the dataframe

sites = folium.map.FeatureGroup()

# circle & marker: loop through the sites and add each to the sites feature group

for lat, lng, lbl in zip(spacex_df.Lat, spacex_df.Long, spacex_df.LaunchSite):
    folium.Circle([lat, lng], radius=300, color='blue', fill=False).add_to(sites)
    folium.Marker([lat, lng], icon=DivIcon(icon_size=(10,10),icon_anchor=(0,0), html='<div style="font-size: 12; color:blue;"><b>%s</b></div>' % lbl, )).add_to(sites)

# add the group to map

site_map.add_child(sites)

Marker cluster

In [221]:
# Let's first create a MarkerCluster object

marker_cluster = MarkerCluster()

# create a function to check the value of `class` column
# If class=1, marker_color value will be green
# If class=0, marker_color value will be red

def color_df(row):
    if row['Class'] == 1:
        return "green"
    else:
        return "red"

spacex_df['MarkerColor'] = spacex_df.apply(color_df, axis=1)

In [ ]:
# for each row in spacex_df_sub, create a Marker object with its coordinate
# and customize the Marker's icon property based on launch success or failure

for lat, lng, clr, otc in zip(spacex_df.Lat, spacex_df.Long, spacex_df.MarkerColor, spacex_df.LandingOutcome):
    folium.Marker([lat, lng], icon=folium.Icon(color='white', icon_color=clr), popup=otc).add_to(marker_cluster)

# Add marker_cluster to current site_map
site_map.add_child(marker_cluster)

# Display the map
site_map

#### Proximities (chosen launch site = VAFB SLC-4E, California)

Prepare the functions

In [223]:
# First to add the mouse position feature: 

formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='bottomleft',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter)

In [224]:
# calculate distances: 

from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # approximate radius of earth in km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

Wrangling & variables

In [225]:
# create group by dataframe to see only the launch sites coordinates: 

launch_sites_df = spacex_df.groupby(['LaunchSite'], as_index=False).first()
launch_sites_df = launch_sites_df[['LaunchSite', 'Lat', 'Long']]

# filter the dataframe and create some variables: 

site = launch_sites_df[launch_sites_df["LaunchSite"] == "VAFB SLC-4E"]
site_lat = site["Lat"].astype("float64")
site_lon = site["Long"].astype("float64")
site_name = site["LaunchSite"].to_list()

site_loc = [site_lat,site_lon]

# using the mouse position function, write down the nearest coordinates: 

coastline_loc = [34.63568,-120.62516]
railway_loc = [34.63727,-120.62345]
city_loc = [34.64947,-120.45771]


In [228]:
# initiliaze new map for the proximities analysis: 

site_map = folium.Map(default_size="10%",location=city_loc, zoom_start=12)

# site_map.add_child(mouse_position)

Polylines

In [ ]:
# now calculate the distance from the VAFB SLC-4E launch site to the nearest coastline using the "calculate_distance" function created before: 

coastline_dist = str(round(calculate_distance(site_lat,site_lon,coastline_loc[0],coastline_loc[1]), ndigits=2)) + " km"
railway_dist = str(round(calculate_distance(site_lat,site_lon,railway_loc[0],railway_loc[1]), ndigits=2)) + " km"
city_dist = str(round(calculate_distance(site_lat,site_lon,city_loc[0],city_loc[1]), ndigits=2)) + " km"

# create two objects - the marker, pointing to chosen point on the coastline, and the polyline, visually connecting the launch site with the nearest coastline: 

site_marker = folium.Marker(location=site_loc, icon=folium.Icon(color = "red", icon_color="white"), popup=site_name[0])
coastline_marker = folium.Marker(location = coastline_loc, icon=folium.Icon(color = "blue", icon_color = "white"), popup=coastline_dist)
railway_marker = folium.Marker(location=railway_loc, icon=folium.Icon(color = "blue", icon_color = "white"), popup=railway_dist)
city_marker = folium.Marker(location=city_loc, icon=folium.Icon(color = "blue", icon_color = "white"), popup=city_dist)

coast_lines=folium.PolyLine(locations=[coastline_loc,site_loc], weight=1)
rail_lines = folium.PolyLine(locations=[railway_loc,site_loc], weight=1)
city_lines = folium.PolyLine(locations=[city_loc,site_loc], weight=1)

site_map.add_child(site_marker)
site_map.add_child(coastline_marker)
site_map.add_child(railway_marker)
site_map.add_child(city_marker)

site_map.add_child(coast_lines)
site_map.add_child(rail_lines)
site_map.add_child(city_lines)